# Unlearning Extraction

Compare base vs unlearned model activations to see if unlearning has a directional bias.
- directional bias -> can try to reverse with ablation
- no delta -> refusal could be epistemic or nonlinear

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datasets import load_dataset
import torch, gc

from src.model_loader import load_model, get_device
from src.hooks import extract_activations_multi
from src.geometry import run_geometry_scan, random_coherence_floor

Load Questions

In [2]:
forget_data = load_dataset("locuslab/TOFU", "forget10")["train"]
forget_questions = [x["question"] for x in forget_data]
print(f"Forget-set questions: {len(forget_questions)}")
print(f"Example: {forget_questions[0]}")

np.random.seed(42)
retain_data = load_dataset("locuslab/TOFU", "retain90")["train"]
retain_idx = np.random.choice(len(retain_data), size=len(forget_questions), replace=False)
retain_questions = [retain_data[int(i)]["question"] for i in retain_idx]

print(f"Retain-set questions: {len(retain_questions)}")
print(f"Example: {retain_questions[0]}")

Forget-set questions: 400
Example: What is the full name of the author born in Taipei, Taiwan on 05/11/1991 who writes in the genre of leadership?
Retain-set questions: 400
Example: What is the full name of this notable science fiction author born in 1934?


Params

In [3]:
CHECKPOINTS = {
    "GradDiff": "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_GradDiff_lr1e-05_alpha5_epoch5",
    "NPO":      "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_NPO_lr1e-05_beta0.5_alpha1_epoch10",
    "AltPO":    "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_AltPO_lr5e-05_beta0.1_alpha1_epoch10",
    "SimNPO":   "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_SimNPO_lr2e-05_b4.5_a1_d1_g0.125_ep10",
    "RMU":      "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_RMU_lr5e-05_layer10_scoeff10_epoch10",
}

RETAIN = "open-unlearning/tofu_Llama-3.2-1B-Instruct_retain90"
BASE_MODEL = "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"

Load models and run scan of difference in activations

In [ ]:
from src.geometry import run_geometry_scan, random_coherence_floor

ALL_LAYERS = [f"model.layers.{i}" for i in range(16)]
ALL_TARGETS = {**CHECKPOINTS, "retain90": RETAIN}

scan = run_geometry_scan(
    base_model_id=BASE_MODEL,
    targets=ALL_TARGETS,
    question_sets={
        "forget": forget_questions,
        "retain": retain_questions,
    },
    layers=ALL_LAYERS,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
Device: mps | dtype: torch.float16
Params: 1.2B


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_GradDiff_lr1e-05_alpha5_epoch5
Device: mps | dtype: torch.float16
Params: 1.2B


Display Results
- difference between methodologies

In [ ]:
ORDER = ["GradDiff", "NPO", "AltPO", "SimNPO", "RMU", "retain90"]

def metric_table(scan_set, metric):
    """scan_set is one question-set slice, e.g. scan['forget']."""
    rows = []
    for layer in ALL_LAYERS:
        idx = int(layer.split(".")[-1])
        row = {"layer": idx}
        for name in ORDER:
            row[name] = scan_set[name][layer][metric]
        rows.append(row)
    return pd.DataFrame(rows)

pd.set_option("display.precision", 3)
print("=== FORGET: Raw coherence ===")
print(metric_table(scan["forget"], "coherence").to_string(index=False))
print("\n=== FORGET: Shift norm ===")
print(metric_table(scan["forget"], "shift_norm").to_string(index=False))
print("\n=== FORGET: Centered coherence ===")
print(metric_table(scan["forget"], "centered").to_string(index=False))

Forget vs Retain Control Shift
- check if fine-tuning drift is responsible for the shifts we see rather than actual unlearning

In [ ]:
import pandas as pd

rows = []
for layer in ALL_LAYERS:
    idx = int(layer.split(".")[-1])
    row = {"layer": idx}
    for name in ["GradDiff", "NPO", "AltPO", "SimNPO", "RMU", "retain90"]:
        f = scan["forget"][name][layer]["shift_norm"]
        r = scan["retain"][name][layer]["shift_norm"]
        row[name] = f / (r + 1e-8)
    rows.append(row)

pd.set_option("display.precision", 2)
print("=== forget-shift / retain-shift ratio (>1 means forget-specific) ===")
print(pd.DataFrame(rows).to_string(index=False))